# 0.8 Authorization Lists And Blob Hashes

This notebook pulls the transaction-content components that are not covered by calldata + BAL:

- EIP-7702 authorization tuples, fetched from JSON-RPC type-4 transactions.
- EIP-4844 blob versioned hashes, counted from Xatu `execution_transaction.blob_hashes`.

The same authorization-list pull feeds two dimensions:

- **Bandwidth / EIP-8131 floor:** every authorization tuple counts, including invalid and duplicate tuples. EIP-8131 uses `108 bytes * 64 gas/byte` per tuple. Blob versioned hashes use `32 bytes * 64 gas/byte`.
- **State / EIP-8037 sensitivity:** only recovered, set-code authorization authorities can become delegation-indicator candidates. Exact state charge still needs pre-state, so this notebook reports an upper-bound sensitivity term.

In [1]:
import os
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.rpc_authorizations import (
    AUTH_TUPLE_BYTES_8131,
    BLOB_VERSIONED_HASH_BYTES_8131,
    fetch_authorization_data_for_blocks,
)
from sim.xatu_calldata import query_xatu_calldata_by_block

pd.options.display.float_format = "{:,.4f}".format

load_dotenv(PROJECT_ROOT / ".env")
missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing .env values: " + ", ".join(missing))

CLICKHOUSE_RAW_HOST = os.environ.get("CLICKHOUSE_RAW_HOST", "clickhouse-raw.xatu.ethpandaops.io")
raw_client = clickhouse_connect.get_client(
    host=CLICKHOUSE_RAW_HOST,
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)

ETHNODEOPS_API_KEY = os.environ.get("ETHNODEOPS_API_KEY") or os.environ.get("hoodi_api_key")
ETHNODEOPS_RPC = os.environ.get("ETHNODEOPS_RPC", "https://erigon.mainnet.rpc.ethnodeops.xyz")
ALCHEMY_RPC = os.environ.get("ALCHEMY_RPC")

if ETHNODEOPS_API_KEY:
    RPC_URL = ETHNODEOPS_RPC
    RPC_HEADERS = {"X-API-Key": ETHNODEOPS_API_KEY}
    RPC_PROVIDER_LABEL = "ethnodeops_erigon_mainnet" if "erigon." in RPC_URL else "ethnodeops_mainnet"
elif ALCHEMY_RPC:
    RPC_URL = ALCHEMY_RPC
    RPC_HEADERS = None
    RPC_PROVIDER_LABEL = "alchemy_mainnet"
else:
    raise RuntimeError("Missing ETHNODEOPS_API_KEY or ALCHEMY_RPC in .env")

print("raw", CLICKHOUSE_RAW_HOST, raw_client.query("SELECT version()").result_rows)
print("rpc_provider", RPC_PROVIDER_LABEL)

raw clickhouse-raw.xatu.ethpandaops.io [('26.2.5.45',)]
rpc_provider ethnodeops_erigon_mainnet


## Parameters

In [2]:
NETWORK = "mainnet"
START_BLOCK = 24_120_001
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
CPSB = 1530

DATA_DIR = PROJECT_ROOT / "data"
AUTH_RECORDS_CSV = DATA_DIR / f"rpc_authorization_records_{min(BLOCKS)}_{max(BLOCKS)}.csv"
AUTH_SUMMARY_CSV = DATA_DIR / f"rpc_authorization_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
CONTENT_CSV = DATA_DIR / f"bandwidth_content_8131_{min(BLOCKS)}_{max(BLOCKS)}.csv"
BAL_CSV = DATA_DIR / f"rpc_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
STATE_CSV = DATA_DIR / f"xatu_state_growth_{min(BLOCKS)}_{max(BLOCKS)}.csv"

WRITE_CSV = True
min(BLOCKS), max(BLOCKS), len(BLOCKS)

(24120001, 24120050, 50)

## Pull Authorization Lists

The helper first queries Xatu for type-4 transaction hashes, then fetches each raw transaction from RPC and decodes the full `authorizationList`. Per-tuple rows are intentionally gross: bandwidth needs every tuple, even if duplicate or invalid.

In [3]:
auth_records, auth_summary = fetch_authorization_data_for_blocks(
    raw_client=raw_client,
    rpc_url=RPC_URL,
    block_numbers=BLOCKS,
    network=NETWORK,
    rpc_headers=RPC_HEADERS,
)

auth_summary["authorization_tuple_8131_floor_gas"] = (
    auth_summary["authorization_tuple_8131_bytes"] * 64
)
auth_summary["authorization_tuple_rlp_floor_gas"] = (
    auth_summary["authorization_tuple_rlp_bytes"] * 64
)
auth_summary["eip8037_delegation_indicator_state_gas_upper_bound"] = (
    auth_summary["authorization_state_upper_bound_authorities"] * 23 * CPSB
)

display(auth_summary.head(12))
display(auth_records.head(12))

if WRITE_CSV:
    DATA_DIR.mkdir(exist_ok=True)
    auth_records.to_csv(AUTH_RECORDS_CSV, index=False)
    auth_summary.to_csv(AUTH_SUMMARY_CSV, index=False)
    print(AUTH_RECORDS_CSV)
    print(AUTH_SUMMARY_CSV)

,block_number,type4_tx_count,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,authorization_set_tuple_count,authorization_clear_tuple_count,authorization_recovered_count,authorization_state_upper_bound_authorities,authorization_tuple_8131_floor_gas,authorization_tuple_rlp_floor_gas,eip8037_delegation_indicator_state_gas_upper_bound
0,24120001,1,1,92,108,1,0,1,1,6912,5888,35190
1,24120002,3,3,276,324,3,0,3,3,20736,17664,105570
2,24120003,1,1,92,108,1,0,1,1,6912,5888,35190
3,24120004,1,1,92,108,1,0,1,1,6912,5888,35190
4,24120005,2,2,184,216,2,0,2,2,13824,11776,70380
5,24120006,3,3,276,324,3,0,3,3,20736,17664,105570
6,24120007,0,0,0,0,0,0,0,0,0,0,0
7,24120008,2,2,184,216,2,0,2,2,13824,11776,70380
8,24120009,1,1,94,108,1,0,1,1,6912,6016,35190
9,24120010,11,11,1034,1188,9,2,11,2,76032,66176,70380


,block_number,tx_index,tx_hash,auth_index,chain_id,target_address,nonce,y_parity,r,s,authority,is_clear,chain_id_valid,nonce_valid,signature_low_s,recovered,recover_error,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes
0,24120001,271,0xcc1925ecdd9726ba0cffbd8c435e93c1f5a26537f445...,0,1,0x0000fb7702036ff9f76044a501ac1aa74cbab16b,0,1,7094542730399798808188250162661520547898673159...,3671797382653276578337153370403359040764547809...,0x358566d044738c064f3a66f8e55c403c71112e2e,False,True,True,True,True,,92,108
1,24120002,43,0x3a0c56c0613343fff6a324587a41de8af72041f3aaa3...,0,1,0xd2e28229f6f2c235e57de2ebc727025a1d0530fb,8,0,1037017594925731419003623322045373558862738383...,4976815638821262260644120757408273254613704339...,0xf0fbbc98a42749ffd446319ac65e7eea460bd6d8,False,True,True,True,True,,92,108
2,24120002,146,0x982b3471be2fb19601f84b4734e44a82f6857b2611e5...,0,1,0x0000fb7702036ff9f76044a501ac1aa74cbab16b,0,1,1047539491018858679206012184996909461921526126...,4445855952410595532506610509651644075726441458...,0x45e55cc3773cf9102e0a97d50e1cd66e81496a2d,False,True,True,True,True,,92,108
3,24120002,147,0xe90819aaf08040e586ed3510ed0b916f92a5e83b1b63...,0,1,0x0000fb7702036ff9f76044a501ac1aa74cbab16b,0,1,9902602702185072352777723123716238864737272542...,5722982953963538148878718894091196787821130034...,0x5fefe433a6d76d7114921503f48c53b9952dfc72,False,True,True,True,True,,92,108
4,24120003,353,0xc6c9521d1254e5e28ffe3750dac573241a5070fe669c...,0,1,0xffc6f9571dc9445a07131476923811daf9621018,3,0,6423519695635598488098800475329770814907532486...,2281393780050507869701849330029766004948252365...,0x78d2bb862a6d90fb6a1e9950189caaac39661ee2,False,True,True,True,True,,92,108
5,24120004,150,0x6108282d35fd629e5f5bfafad40c16d1116f09f301f5...,0,1,0xffc6f9571dc9445a07131476923811daf9621018,10,1,5387727440015798120348682967434441839170263087...,3875631564511763097951663485775730849323524821...,0xa7a4091fff3ca4aec93bc5bd63e45e09545217ac,False,True,True,True,True,,92,108
6,24120005,26,0x63c1214f6fa42b429ce7fd87e3556faba06f4e2ce01a...,0,1,0x4884d28f048e66a537762334937e01a044cbdfac,1,0,1047584364097725488525952006326185049176709560...,2391522499106050365838625633719354022653061744...,0xb953e72085177d03d99b855e1dc2dbb6ec3d57c7,False,True,True,True,True,,92,108
7,24120005,170,0x1ac613efc3c9c8fe6d5f5e4bff86849aaf0ece85d26c...,0,1,0xffc6f9571dc9445a07131476923811daf9621018,2,1,9339792010882719898682714429281318202994470234...,2856798006254249289551668517206134582660179499...,0x02170b5401d26436738b18333e5b85f73f1e2a1c,False,True,True,True,True,,92,108
8,24120006,45,0x995f55df6cff0e275615be433af73ef3756e3c1698b8...,0,1,0x7702cb554e6bfb442cb743a7df23154544a7176c,36,0,2744337722887222608413930692428781815363239545...,3974397195157245428718351630571363126490383401...,0x95910ba86d22efc77197246cdd17b11b78bb0d78,False,True,True,True,True,,92,108
9,24120006,59,0x2b1608c71a0f023bbe7193eb8f9c4e2875fa1ab12a52...,0,1,0xd2e28229f6f2c235e57de2ebc727025a1d0530fb,0,1,6703385683294952821777376165528818205993566401...,1425419666736699273284358983242795537813802775...,0x91847abfbb976381c03961849d6fbeda2b2f8519,False,True,True,True,True,,92,108


/Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_records_24120001_24120050.csv
/Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_summary_24120001_24120050.csv


## Pull Blob Versioned Hash Counts

Blob versioned hashes are already in Xatu `execution_transaction.blob_hashes`. EIP-8131 treats each hash as 32 bytes at the 64 gas/byte floor.

In [4]:
calldata = query_xatu_calldata_by_block(raw_client, BLOCKS, network=NETWORK)
blob_cols = [
    "block_number",
    "calldata_bytes",
    "calldata_zero_bytes",
    "calldata_nonzero_bytes",
    "calldata_gas_7999",
    "blob_versioned_hash_count",
    "blob_versioned_hash_bytes",
    "blob_versioned_hash_8131_floor_gas",
]
display(calldata[blob_cols].head(12))

,block_number,calldata_bytes,calldata_zero_bytes,calldata_nonzero_bytes,calldata_gas_7999,blob_versioned_hash_count,blob_versioned_hash_bytes,blob_versioned_hash_8131_floor_gas
0,24120001,238525,144495,94030,2082460,13,416,26624
1,24120002,44492,29852,14640,353648,6,192,12288
2,24120003,136567,86377,50190,1148548,4,128,8192
3,24120004,70460,44058,26402,598664,9,288,18432
4,24120005,46996,30210,16786,389416,0,0,0
5,24120006,101974,57426,44548,942472,5,160,10240
6,24120007,12220,6755,5465,114460,1,32,2048
7,24120008,133308,89789,43519,1055460,8,256,16384
8,24120009,91881,62112,29769,724752,0,0,0
9,24120010,145880,81120,64760,1360640,5,160,10240


## Join Into Bandwidth And State Add-On Table

`bandwidth_content_bytes_actual_auth` uses actual per-tuple RLP bytes. `bandwidth_content_bytes_8131_auth` uses the EIP-8131 constant `108 bytes` per authorization tuple. The latter is the right column for EIP-8131 floor-gas calculations.

In [5]:
content = calldata[blob_cols].merge(auth_summary, on="block_number", how="left")
for column in [
    "type4_tx_count",
    "authorization_tuple_count",
    "authorization_tuple_rlp_bytes",
    "authorization_tuple_8131_bytes",
    "authorization_set_tuple_count",
    "authorization_clear_tuple_count",
    "authorization_recovered_count",
    "authorization_state_upper_bound_authorities",
    "authorization_tuple_8131_floor_gas",
    "authorization_tuple_rlp_floor_gas",
    "eip8037_delegation_indicator_state_gas_upper_bound",
]:
    content[column] = content[column].fillna(0).astype("int64")

if BAL_CSV.exists():
    bal = pd.read_csv(BAL_CSV)[["block_number", "bal_rlp_bytes"]]
    content = content.merge(bal, on="block_number", how="left")
else:
    content["bal_rlp_bytes"] = 0
content["bal_rlp_bytes"] = content["bal_rlp_bytes"].fillna(0).astype("int64")

content["bandwidth_content_bytes_actual_auth"] = (
    content["calldata_bytes"]
    + content["bal_rlp_bytes"]
    + content["authorization_tuple_rlp_bytes"]
    + content["blob_versioned_hash_bytes"]
)
content["bandwidth_content_bytes_8131_auth"] = (
    content["calldata_bytes"]
    + content["bal_rlp_bytes"]
    + content["authorization_tuple_8131_bytes"]
    + content["blob_versioned_hash_bytes"]
)
content["eip8131_auth_blob_floor_gas"] = (
    content["authorization_tuple_8131_floor_gas"]
    + content["blob_versioned_hash_8131_floor_gas"]
)

if STATE_CSV.exists():
    state = pd.read_csv(STATE_CSV)[["block_number", "state_gas_used"]]
    content = content.merge(state, on="block_number", how="left")
    content["state_gas_with_auth_indicator_upper_bound"] = (
        content["state_gas_used"].fillna(0).astype("int64")
        + content["eip8037_delegation_indicator_state_gas_upper_bound"]
    )
else:
    content["state_gas_used"] = pd.NA
    content["state_gas_with_auth_indicator_upper_bound"] = pd.NA

display(content.head(12))

if WRITE_CSV:
    content.to_csv(CONTENT_CSV, index=False)
    print(CONTENT_CSV)

,block_number,calldata_bytes,calldata_zero_bytes,calldata_nonzero_bytes,calldata_gas_7999,blob_versioned_hash_count,blob_versioned_hash_bytes,blob_versioned_hash_8131_floor_gas,type4_tx_count,authorization_tuple_count,...,authorization_state_upper_bound_authorities,authorization_tuple_8131_floor_gas,authorization_tuple_rlp_floor_gas,eip8037_delegation_indicator_state_gas_upper_bound,bal_rlp_bytes,bandwidth_content_bytes_actual_auth,bandwidth_content_bytes_8131_auth,eip8131_auth_blob_floor_gas,state_gas_used,state_gas_with_auth_indicator_upper_bound
0,24120001,238525,144495,94030,2082460,13,416,26624,1,1,...,1,6912,5888,35190,339327,578360,578376,33536,42275430,42310620
1,24120002,44492,29852,14640,353648,6,192,12288,3,3,...,3,20736,17664,105570,72309,117269,117317,33024,13623120,13728690
2,24120003,136567,86377,50190,1148548,4,128,8192,1,1,...,1,6912,5888,35190,186956,323743,323759,15104,43098570,43133760
3,24120004,70460,44058,26402,598664,9,288,18432,1,1,...,1,6912,5888,35190,117839,188679,188695,25344,31432320,31467510
4,24120005,46996,30210,16786,389416,0,0,0,2,2,...,2,13824,11776,70380,81624,128804,128836,13824,13523670,13594050
5,24120006,101974,57426,44548,942472,5,160,10240,3,3,...,3,20736,17664,105570,129883,232293,232341,30976,85699890,85805460
6,24120007,12220,6755,5465,114460,1,32,2048,0,0,...,0,0,0,0,19240,31492,31492,2048,4430880,4430880
7,24120008,133308,89789,43519,1055460,8,256,16384,2,2,...,2,13824,11776,70380,175656,309404,309436,30208,58289940,58360320
8,24120009,91881,62112,29769,724752,0,0,0,1,1,...,1,6912,6016,35190,109312,201287,201301,6912,52833960,52869150
9,24120010,145880,81120,64760,1360640,5,160,10240,11,11,...,2,76032,66176,70380,169659,316733,316887,86272,77177790,77248170


/Users/william/PycharmProjects/eip-7999-research/data/bandwidth_content_8131_24120001_24120050.csv


## Summary

In [6]:
summary = pd.DataFrame(
    [
        {
            "blocks": len(content),
            "type4_tx_count": int(content["type4_tx_count"].sum()),
            "authorization_tuple_count": int(content["authorization_tuple_count"].sum()),
            "authorization_tuple_rlp_bytes": int(content["authorization_tuple_rlp_bytes"].sum()),
            "authorization_tuple_8131_bytes": int(content["authorization_tuple_8131_bytes"].sum()),
            "authorization_tuple_8131_floor_gas": int(content["authorization_tuple_8131_floor_gas"].sum()),
            "blob_versioned_hash_count": int(content["blob_versioned_hash_count"].sum()),
            "blob_versioned_hash_bytes": int(content["blob_versioned_hash_bytes"].sum()),
            "blob_versioned_hash_8131_floor_gas": int(content["blob_versioned_hash_8131_floor_gas"].sum()),
            "eip8131_auth_blob_floor_gas": int(content["eip8131_auth_blob_floor_gas"].sum()),
            "authorization_state_upper_bound_authorities": int(content["authorization_state_upper_bound_authorities"].sum()),
            "eip8037_delegation_indicator_state_gas_upper_bound": int(content["eip8037_delegation_indicator_state_gas_upper_bound"].sum()),
            "bandwidth_content_bytes_actual_auth": int(content["bandwidth_content_bytes_actual_auth"].sum()),
            "bandwidth_content_bytes_8131_auth": int(content["bandwidth_content_bytes_8131_auth"].sum()),
        }
    ]
)

display(summary)

,blocks,type4_tx_count,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,authorization_tuple_8131_floor_gas,blob_versioned_hash_count,blob_versioned_hash_bytes,blob_versioned_hash_8131_floor_gas,eip8131_auth_blob_floor_gas,authorization_state_upper_bound_authorities,eip8037_delegation_indicator_state_gas_upper_bound,bandwidth_content_bytes_actual_auth,bandwidth_content_bytes_8131_auth
0,50,67,67,6225,7236,463104,235,7520,481280,944384,51,1794690,12689704,12690715
